# Augmentation-utility analysis of GenAI-generated reviews

Synthetic reviews for all eight Plutchik emotions (**Anger**, **Anticipation**, **Disgust**, **Fear**, **Joy**, **Sadness**, **Surprise**, **Trust**) produced by three GenAI models (`Claude`, `Gemini`, `GPT` / OpenAI) under three prompting strategies (`zero-shot`, `few-shot (guidelines)`, `few-shot (guidelines + dataset)`).

Data layout under `data/stynthetic/`:

```
{Claude|Gemini|OpenAi}/
  {0Shoot|FewShoot|FewShootExample[s]}/
    generated_{provider}_{emotion}_reviews_{strategy}.csv
```

**Decision**: pick the best `(Strategy, GenAI)` combination for **augmenting** the training corpus of an encoder-only emotion classifier on under-represented emotions.

**Criteria** (computed per emotion, on 384-d sentence embeddings, cosine):

| Metric | Direction | Maps to |
|---|---|---|
| `diversity_intra` — mean pairwise distance *within* synth | ↑ | covers the emotion spectrum |
| `novelty_vs_synth` — mean nearest-neighbour distance *within* synth | ↑ | no near-duplicates among synth |
| `novelty_vs_gt` — mean distance from each synth sample to its nearest GT sample | ↑ | not a paraphrase of existing GT |
| `on_emotion` — mean cosine similarity of synth to the GT emotion centroid | ↑ | synth is still labelled correctly |

A composite **Borda rank** over the four metrics yields a single ranking per emotion, plus a macro-averaged ranking across emotions.

In [24]:
from __future__ import annotations

import hashlib
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_distances

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / "data").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR      = REPO_ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "stynthetic"
CACHE_DIR     = REPO_ROOT / "code" / "data_augmentation" / ".embeddings_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Plutchik basic emotions (exclude Neutral / Reject).
EMOTIONS = [
    "Anger", "Anticipation", "Disgust", "Fear",
    "Joy", "Sadness", "Surprise", "Trust",
]

# Folder → display-name maps for the new tree layout.
GENAI_DIR_MAP = {
    "Claude": "Claude",
    "Gemini": "Gemini",
    "OpenAi": "GPT",
}
STRATEGY_DIR_MAP = {
    "0Shoot": "zero-shot",
    "FewShoot": "few-shot (guidelines)",
    "FewShootExample": "few-shot (guidelines + dataset)",
    "FewShootExamples": "few-shot (guidelines + dataset)",  # Claude spelling
}

print("Repo root:", REPO_ROOT)
print("Synthetic dir:", SYNTHETIC_DIR)
print("Emotions:", ", ".join(EMOTIONS))

Repo root: c:\Users\joaquim.motger\Projectes\review-emotion-analysis


## 1. Load datasets

Ground-truth annotations are **multi-label**, so we use independent set-membership: a GT sentence belongs to emotion `e` if `e == 1`, regardless of any other emotion tags it may also carry.

Synthetic CSVs are discovered under `data/stynthetic/{GenAI}/{Strategy}/`. Emotion is taken from the filename; text is the `sentence` column (aligned with GT).

In [25]:
gt_raw = pd.read_excel(DATA_DIR / "ground-truth.xlsx")
missing_gt = [e for e in EMOTIONS if e not in gt_raw.columns]
if missing_gt:
    raise KeyError(f"GT file missing emotion columns: {missing_gt}")

gt = gt_raw[["sentence", *EMOTIONS]].copy()
gt["sentence"] = gt["sentence"].astype(str).str.strip()
gt = gt[gt["sentence"].str.len() > 0].reset_index(drop=True)

gt_by_emotion = {e: gt.loc[gt[e] == 1, "sentence"].tolist() for e in EMOTIONS}

_EMOTION_RE = re.compile(
    r"generated_[^_]+_([a-z]+)_reviews_",
    re.IGNORECASE,
)
_EMOTION_LOOKUP = {e.lower(): e for e in EMOTIONS}


def _emotion_from_filename(path: Path) -> str | None:
    m = _EMOTION_RE.search(path.name)
    if not m:
        return None
    return _EMOTION_LOOKUP.get(m.group(1).lower())


def _load_synth_csv(path: Path, emotion: str, strategy: str, genai: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [c.strip() for c in df.columns]
    # Prefer sentence (matches GT); fall back to review if needed.
    text_col = "sentence" if "sentence" in df.columns else "review" if "review" in df.columns else None
    if text_col is None:
        raise KeyError(f"No sentence/review column in {path}")

    out = pd.DataFrame({
        "Review": df[text_col].astype(str).str.strip().str.strip('"').str.strip(),
        "Strategy": strategy,
        "GenAI": genai,
        "Emotion": emotion,
        "source_file": path.name,
    })
    return out[out["Review"].str.len() > 0].reset_index(drop=True)


def load_all_synthetic(root: Path) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    skipped: list[str] = []

    for genai_dir, genai_name in GENAI_DIR_MAP.items():
        provider_root = root / genai_dir
        if not provider_root.is_dir():
            skipped.append(f"missing provider dir: {genai_dir}")
            continue

        for strategy_dir in sorted(p.name for p in provider_root.iterdir() if p.is_dir()):
            strategy = STRATEGY_DIR_MAP.get(strategy_dir)
            if strategy is None:
                skipped.append(f"unknown strategy dir: {genai_dir}/{strategy_dir}")
                continue

            for csv_path in sorted((provider_root / strategy_dir).glob("*.csv")):
                emotion = _emotion_from_filename(csv_path)
                if emotion is None:
                    skipped.append(f"unrecognised emotion file: {csv_path.name}")
                    continue
                frames.append(_load_synth_csv(csv_path, emotion, strategy, genai_name))

    if not frames:
        raise FileNotFoundError(f"No synthetic CSVs found under {root}")

    synth_df = pd.concat(frames, ignore_index=True)
    if skipped:
        print(f"Skipped {len(skipped)} path(s):")
        for s in skipped:
            print(f"  - {s}")
    return synth_df


synth = load_all_synthetic(SYNTHETIC_DIR)

print("Ground truth sentences:")
for e in EMOTIONS:
    print(f"  {e:13s}: {len(gt_by_emotion[e])}")

print(f"\nSynthetic rows loaded: {len(synth)}")
print("Synthetic samples by Emotion × Strategy × GenAI:")
counts = synth.groupby(["Emotion", "Strategy", "GenAI"]).size().unstack(fill_value=0)
# Stable column / emotion order for readability.
counts = counts.reindex(columns=["Claude", "Gemini", "GPT"], fill_value=0)
counts = counts.reindex(
    pd.MultiIndex.from_product(
        [EMOTIONS, ["zero-shot", "few-shot (guidelines)", "few-shot (guidelines + dataset)"]],
        names=["Emotion", "Strategy"],
    ),
    fill_value=0,
)
display(counts)

Ground truth sentences:
  Fear     : 24
  Surprise : 54

Synthetic samples by Emotion × Strategy × GenAI:
GenAI                                     Claude  GPT  Gemini
Emotion  Strategy                                            
Fear     few-shot (guidelines + dataset)      20   20      20
         few-shot (guidelines)                20   20      20
         zero-shot                            20   20      20
Surprise few-shot (guidelines + dataset)      20   20      20
         few-shot (guidelines)                20   20      20
         zero-shot                            20   20      20


## 2. Embed with an encoder-only LLM

We use [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) — a 6-layer encoder-only MiniLM (22M params, 384-dim), fast on CPU and well-suited to short reviews. Embeddings are L2-normalised (so cosine similarity = dot product) and cached on disk.

In [26]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(EMBED_MODEL_NAME)
EMBED_DIM = encoder.get_sentence_embedding_dimension()
print(f"Loaded {EMBED_MODEL_NAME} (dim={EMBED_DIM})")


def _embed(texts: list[str], tag: str) -> np.ndarray:
    h = hashlib.sha1(EMBED_MODEL_NAME.encode())
    for t in texts:
        h.update(b"\x1f"); h.update(t.encode("utf-8", errors="ignore"))
    cache = CACHE_DIR / f"{tag}_{h.hexdigest()}.npy"
    if cache.exists():
        return np.load(cache)
    emb = encoder.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(cache, emb)
    return emb


gt_emb = {e: _embed(gt_by_emotion[e], f"gt_{e}") for e in EMOTIONS}
synth_emb = _embed(synth["Review"].tolist(), "synth_all")

for e in EMOTIONS:
    print(f"  GT {e:13s} embeddings: {gt_emb[e].shape}")
print(f"  Synth embeddings              : {synth_emb.shape}")
print(f"  Combinations expected         : "
      f"{len(EMOTIONS)} emotions × 3 strategies × 3 models = {len(EMOTIONS) * 9}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3544.96it/s]


Loaded sentence-transformers/all-MiniLM-L6-v2 (dim=384)


Batches: 100%|██████████| 6/6 [00:02<00:00,  2.84it/s]

  GT Fear     embeddings: (24, 384)
  GT Surprise embeddings: (54, 384)
  Synth embeddings   : (360, 384)


## 3. Augmentation-utility metrics

For each emotion `e` and each `(Strategy, GenAI)` group of synthetic samples:

- `diversity_intra` — mean of the upper triangle of the within-synth cosine-distance matrix.
- `novelty_vs_synth` — for each synth point, distance to its nearest other synth point; averaged.
- `novelty_vs_gt` — for each synth point, distance to its nearest GT-`e` point; averaged.
- `on_emotion` — cosine similarity of synth samples to the L2-normalised GT-`e` centroid; averaged.

All four are **higher = better** for augmentation purposes. The composite **Borda rank** sums each metric's per-row rank (higher rank = better); the top row is the recommended generator for that emotion.

In [27]:
def _centroid(x: np.ndarray) -> np.ndarray:
    c = x.mean(axis=0)
    n = np.linalg.norm(c)
    return c / n if n > 0 else c


def _augmentation_metrics(s_emb: np.ndarray, gt_emb_e: np.ndarray) -> dict:
    n = s_emb.shape[0]
    if n < 2 or gt_emb_e.shape[0] == 0:
        return {
            "diversity_intra": np.nan,
            "novelty_vs_synth": np.nan,
            "novelty_vs_gt": np.nan,
            "on_emotion": np.nan,
        }

    d_ss = cosine_distances(s_emb, s_emb)
    iu = np.triu_indices_from(d_ss, k=1)
    diversity_intra = float(d_ss[iu].mean())

    np.fill_diagonal(d_ss, np.inf)
    novelty_vs_synth = float(d_ss.min(axis=1).mean())

    d_sg = cosine_distances(s_emb, gt_emb_e)
    novelty_vs_gt = float(d_sg.min(axis=1).mean())

    on_emotion = float((s_emb @ _centroid(gt_emb_e)).mean())

    return {
        "diversity_intra": diversity_intra,
        "novelty_vs_synth": novelty_vs_synth,
        "novelty_vs_gt": novelty_vs_gt,
        "on_emotion": on_emotion,
    }


rows = []
for (emotion, strategy, genai), idx in synth.groupby(
    ["Emotion", "Strategy", "GenAI"]
).groups.items():
    sub_emb = synth_emb[list(idx)]
    m = _augmentation_metrics(sub_emb, gt_emb[emotion])
    rows.append({
        "Emotion": emotion,
        "Strategy": strategy,
        "GenAI": genai,
        "n": len(idx),
        **m,
    })

metrics_df = pd.DataFrame(rows)
# Stable sort for readability before ranking.
metrics_df = metrics_df.sort_values(
    ["Emotion", "Strategy", "GenAI"]
).reset_index(drop=True)
print(f"Computed metrics for {len(metrics_df)} (Emotion, Strategy, GenAI) groups "
      f"(expected {len(EMOTIONS) * 9}).")
display(metrics_df.round(4))

Computed metrics for 18 (Emotion, Strategy, GenAI) groups.


In [29]:
METRIC_COLS = ["diversity_intra", "novelty_vs_synth", "novelty_vs_gt", "on_emotion"]


def _ranked_table(df_e: pd.DataFrame) -> pd.DataFrame:
    df_e = df_e.copy()
    # Per-metric rank: higher value -> higher rank (best = highest rank).
    for c in METRIC_COLS:
        df_e[f"rk_{c}"] = df_e[c].rank(method="average")
    df_e["borda"] = df_e[[f"rk_{c}" for c in METRIC_COLS]].sum(axis=1)
    df_e = df_e.sort_values("borda", ascending=False).reset_index(drop=True)
    df_e.insert(0, "overall_rank", df_e.index + 1)
    return df_e


per_emotion_winners = []
for e in EMOTIONS:
    print(f"\n=== {e} — augmentation utility ranking "
          f"(top = best for fine-tuning augmentation) ===")
    table = _ranked_table(metrics_df[metrics_df["Emotion"] == e])
    display_cols = ["overall_rank", "Strategy", "GenAI", "n", *METRIC_COLS, "borda"]
    display(table[display_cols].round(4))
    top = table.iloc[0]
    per_emotion_winners.append({
        "Emotion": e,
        "Strategy": top["Strategy"],
        "GenAI": top["GenAI"],
        "borda": top["borda"],
        "n": top["n"],
    })

print("\n=== Per-emotion winners (overall_rank == 1) ===")
display(pd.DataFrame(per_emotion_winners))


=== Fear — augmentation utility ranking (top = best for fine-tuning augmentation) ===


,overall_rank,Strategy,GenAI,n,diversity_intra,borda
0,1,few-shot (guidelines),Gemini,20,0.7967,9.0
1,2,few-shot (guidelines),Claude,20,0.7635,8.0
2,3,few-shot (guidelines + dataset),Gemini,20,0.7383,7.0
3,4,few-shot (guidelines + dataset),Claude,20,0.7215,6.0
4,5,zero-shot,Gemini,20,0.7204,5.0
5,6,zero-shot,Claude,20,0.7163,4.0
6,7,few-shot (guidelines + dataset),GPT,20,0.6631,3.0
7,8,few-shot (guidelines),GPT,20,0.5769,2.0
8,9,zero-shot,GPT,20,0.4752,1.0



=== Surprise — augmentation utility ranking (top = best for fine-tuning augmentation) ===


,overall_rank,Strategy,GenAI,n,diversity_intra,borda
0,1,zero-shot,Gemini,20,0.7797,9.0
1,2,few-shot (guidelines),Gemini,20,0.7790,8.0
2,3,few-shot (guidelines + dataset),Gemini,20,0.7693,7.0
3,4,few-shot (guidelines + dataset),GPT,20,0.7559,6.0
4,5,few-shot (guidelines + dataset),Claude,20,0.7424,5.0
5,6,zero-shot,Claude,20,0.7143,4.0
6,7,few-shot (guidelines),GPT,20,0.7032,3.0
7,8,few-shot (guidelines),Claude,20,0.6875,2.0
8,9,zero-shot,GPT,20,0.5171,1.0


In [ ]:
print("\n=== Aggregate across all emotions — macro-averaged augmentation utility ===")
agg = (
    metrics_df
      .groupby(["Strategy", "GenAI"], as_index=False)
      .agg(
          emotions_covered=("Emotion", "nunique"),
          n_total=("n", "sum"),
          **{c: (c, "mean") for c in METRIC_COLS},
      )
)
agg_table = _ranked_table(agg)
display(agg_table[
    ["overall_rank", "Strategy", "GenAI", "emotions_covered", "n_total",
     *METRIC_COLS, "borda"]
].round(4))

winner = agg_table.iloc[0]
print(
    f"\nRecommended single pipeline: "
    f"{winner['GenAI']} + {winner['Strategy']} "
    f"(borda={winner['borda']:.1f}, emotions_covered={int(winner['emotions_covered'])})"
)

# How often does the aggregate winner also win per emotion?
agg_key = (winner["Strategy"], winner["GenAI"])
n_match = sum(
    1 for w in per_emotion_winners
    if (w["Strategy"], w["GenAI"]) == agg_key
)
print(
    f"Aggregate winner is also per-emotion winner in "
    f"{n_match}/{len(per_emotion_winners)} emotions."
)

## 4. How to read the tables — and pick a winner

- **Best per emotion**: the row with `overall_rank == 1` in each per-emotion table is the recommended `(Strategy, GenAI)` for augmenting that emotion's training data. The summary table of winners makes emotion-specific choices easy to scan.
- **Best overall (cross-emotion)**: the row with `overall_rank == 1` in the aggregate table is the recommendation when you want a *single* pipeline that works across all emotions. The aggregate uses a **macro-average** (each emotion contributes equally regardless of sample count / GT size), which matches generalising to other under-represented emotions later.
- **If two cells are close on `borda`**, inspect per-metric trade-offs. Typical patterns:
  - High `novelty_vs_gt` but low `on_emotion` → samples have drifted off the emotion's semantic region; the label may be noisy.
  - High `diversity_intra` but low `novelty_vs_synth` → samples spread out on average but the set still contains some near-duplicates; you may want to deduplicate before training.
  - High `on_emotion` but low `novelty_vs_gt` → samples paraphrase the GT closely; safe but adds little new information.
- **Cross-emotion stability check**: if the aggregate winner is also frequent among per-emotion winners, the choice is robust. If it wins on average but ranks poorly for specific emotions, prefer emotion-specific pipelines for those cases.

### Caveats

- Sample sizes per cell are modest (~20 synthetic samples / cell). Borda ranks are signals, not significance tests.
- All metrics are based on 384-d MiniLM embeddings — a different encoder (e.g. the one you'll fine-tune) may reorder things at the margin.
- `on_emotion` is anchored on the GT centroid; emotions with very few GT positives (e.g. Fear, Anger) have noisier centroids than well-represented ones (e.g. Joy, Sadness).
- Synthetic text uses the `sentence` column to stay aligned with GT sentence-level labels.
- Folder spelling differs slightly across providers (`FewShootExample` vs `FewShootExamples`); both map to the same strategy label.